# HydroDenoise-TF — DCAMF-Net 完整实验流程

本 Notebook 按步骤复现论文中的全部实验：数据准备 → 训练 → 推理 → 基线 → 消融 → 可视化。

**运行前确保**：已按照 README 配置好 conda 环境和数据集。

---
## Part 1: 环境检查与数据准备

### 1.1 环境检查

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "8"  # suppress AutoDL warning

import torch, torchaudio, numpy as np, soundfile as sf
import matplotlib.pyplot as plt
from pathlib import Path
import sys, subprocess, json, re, random

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PROJECT_ROOT = Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "dcamf_net"))  # for bare imports
print(f"Project root: {PROJECT_ROOT}")


### 1.2 检查原始数据

In [ ]:
raw_dir = PROJECT_ROOT / "raw_data"
for ds in ["ShipsEar", "DeepShip"]:
    ds_dir = raw_dir / ds
    if ds_dir.exists():
        wavs = list(ds_dir.rglob("*.wav"))
        print(f"{ds}: {len(wavs)} WAV files found")
        for sub in sorted(ds_dir.iterdir()):
            if sub.is_dir():
                print(f"  {sub.name}/ -> {len(list(sub.glob('*.wav')))} files")
    else:
        print(f"{ds}: NOT FOUND - please download to {ds_dir}")


### 1.3 生成训练/测试数据

In [ ]:
result = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_data.py")],
    cwd=str(PROJECT_ROOT / "scripts"), capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


### 1.4 数据预览

In [ ]:
data_dir = PROJECT_ROOT / "data" / "ShipsEar"
for subset in ["train", "test1", "test2", "test3"]:
    clean_dir = data_dir / subset / "clean"
    if clean_dir.exists():
        files = list(clean_dir.glob("*.wav"))
        print(f"{subset}: {len(files)} samples")

# Load a sample
import soundfile as sf
train_clean = sorted(list((data_dir / "train" / "clean").glob("*.wav")))
train_noisy_dir = data_dir / "train" / "noisy"
sample_clean, sr = sf.read(str(train_clean[0]))
sample_noisy, _ = sf.read(str(train_noisy_dir / train_clean[0].name))
print(f"Sample shape: {sample_clean.shape}, SR: {sr}")

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
t = np.arange(len(sample_clean)) / sr
axes[0].plot(t, sample_noisy, alpha=0.6, label="Noisy", color="gray")
axes[0].plot(t, sample_clean, label="Clean", color="black")
axes[0].set_xlim(0, 0.1); axes[0].legend(); axes[0].set_title("Time Domain (first 100ms)")

from scipy.signal import welch
f, pxx_clean = welch(sample_clean, sr, nperseg=1024)
_, pxx_noisy = welch(sample_noisy, sr, nperseg=1024)
axes[1].semilogy(f, pxx_noisy, alpha=0.6, color="gray", label="Noisy")
axes[1].semilogy(f, pxx_clean, color="black", label="Clean")
axes[1].set_xlim(0, 4000); axes[1].legend(); axes[1].set_title("Power Spectral Density")
plt.tight_layout(); plt.show()


---
## Part 2: DCAMF-Net 训练

### 2.1 模型结构 + 参数量

In [ ]:
from dcamf_net.model import DCAMFNet
from dcamf_net.config import build_model
from torchinfo import summary
from argparse import Namespace

args = Namespace(
    enc_channels=256, enc_kernel_size=80, enc_stride=40,
    chunk_size=500, hop_size=250, n_blocks=10, n_heads=8,
    ffn_hidden=512, dw_kernel_size=31, checkpoint=None
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(args, device=device)
model.eval()

print(summary(model, input_size=(1, 1, 48000), depth=6, col_names=("input_size", "output_size", "num_params")))

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print()
print(f"Total: {total_params:,} params | Trainable: {trainable_params:,}")

try:
    from thop import profile
    dummy = torch.randn(1, 1, 48000).to(device)
    flops, _ = profile(model, inputs=(dummy,), verbose=False)
    print(f"FLOPs: {flops/1e9:.2f} G")
except Exception as e:
    print(f"FLOPs skipped: {e}")


### 2.2 训练 DCAMF-Net

In [ ]:
import platform, subprocess
train_script = str(PROJECT_ROOT / "dcamf_net" / "train.py")
train_dir = str(PROJECT_ROOT / "data" / "ShipsEar" / "train")
save_dir = str(PROJECT_ROOT / "experiments" / "dcamf_net" / "checkpoints")

cmd = [sys.executable, train_script,
       "--train_dir", train_dir,
       "--lr", "5e-4",
       "--save_dir", save_dir,
       "--use_tensorboard"]

print("Training command (works on Windows & Linux):")
print(" ".join(cmd))
print()
print("Run the cell below to start training, or paste the command in terminal.")


### 2.3 训练曲线

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
curve_path = PROJECT_ROOT / "experiments" / "dcamf_net" / "checkpoints" / "learning_curves.png"
if curve_path.exists():
    img = Image.open(curve_path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img); plt.axis('off'); plt.show()
else:
    print("Training not complete yet. Curves saved automatically to:")
    print(f"  {curve_path}")


---
## Part 3: DCAMF-Net 推理

### 3.1 加载模型

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(args, device=device)
ckpt_path = PROJECT_ROOT / "experiments" / "dcamf_net" / "checkpoints" / "best_SISNR.pth"

if ckpt_path.exists():
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    # Filter out non-parameter keys (total_ops, total_params, mask_fusion_weights)
    state_dict = {k: v for k, v in checkpoint.items()
                   if not k.endswith((".total_ops", ".total_params"))
                   and k != "mask_fusion_weights"
                   and k != "total_ops"
                   and k != "total_params"}
    model.load_state_dict(state_dict)
    model.eval()
    print("Model loaded successfully from best_SISNR.pth")
else:
    print(f"NOT FOUND: {ckpt_path}")


### 3.2 各测试集指标

In [ ]:
import pandas as pd

results = {
    "Model": ["CRN", "Conv-TasNet", "DPRNN", "DCAMF-Net (Ours)"],
    "ShipsEar SI-SNRi (dB)": [6.27, 13.65, 7.67, 12.13],
    "ShipsEar SDRi (dB)": [9.03, -7.56, 11.72, 12.84],
    "DeepShip SI-SNRi (dB)": [2.09, 2.05, 0.30, 2.76],
    "DeepShip SDRi (dB)": [-10.34, -23.50, 0.57, 5.57],
}
df = pd.DataFrame(results)
def highlight_best(s):
    return ["font-weight: bold; background-color: #D5F0E0" if v == s.max() else "" for v in s]
display(df.style.apply(highlight_best, subset=[c for c in df.columns if "dB" in c]))


### 3.3 降噪试听 + 波形可视化

In [ ]:
import IPython.display as ipd

denoised_dir = PROJECT_ROOT / "experiments" / "dcamf_net" / "denoised" / "ShipsEar_test1"
clean_dir = PROJECT_ROOT / "data" / "ShipsEar" / "test1" / "clean"
noisy_dir = PROJECT_ROOT / "data" / "ShipsEar" / "test1" / "noisy"

if denoised_dir.exists():
    files = sorted(list(denoised_dir.glob("*.wav")))
    f = random.choice(files)
    clean, sr = sf.read(str(clean_dir / f.name))
    noisy, _ = sf.read(str(noisy_dir / f.name))
    enhanced, _ = sf.read(str(f))
    min_len = min(len(clean), len(noisy), len(enhanced))
    clean, noisy, enhanced = clean[:min_len], noisy[:min_len], enhanced[:min_len]

    fig, axes = plt.subplots(3, 1, figsize=(14, 7))
    t = np.arange(min_len) / sr
    for ax, sig, title, color in zip(axes, [clean, noisy, enhanced],
        ["Clean", "Noisy Input", "DCAMF-Net Enhanced"], ['black', 'gray', '#1C7293']):
        ax.plot(t, sig, color=color, linewidth=0.6); ax.set_title(title); ax.set_xlim(0, 0.2)
    plt.tight_layout(); plt.show()

    print("Noisy:"); ipd.display(ipd.Audio(noisy, rate=sr))
    print("Enhanced:"); ipd.display(ipd.Audio(enhanced, rate=sr))
    print("Clean:"); ipd.display(ipd.Audio(clean, rate=sr))
else:
    print("Denoised directory not found. Run inference (dcamf_net/test.py) first.")


### 3.4 PSD 频谱对比

In [ ]:
from scipy.signal import welch

def psd_db(sig, sr, nperseg=1024):
    f, pxx = welch(sig, sr, nperseg=nperseg, noverlap=nperseg//2)
    return f, 10 * np.log10(pxx + 1e-10)

if denoised_dir.exists():
    fig, ax = plt.subplots(figsize=(10, 5))
    f_c, p_c = psd_db(clean, sr)
    f_n, p_n = psd_db(noisy, sr)
    f_e, p_e = psd_db(enhanced, sr)
    mask = (f_c >= 0) & (f_c <= 4000)
    ax.plot(f_c[mask]/1000, p_c[mask], 'k-', lw=1.5, label="Clean")
    ax.plot(f_n[mask]/1000, p_n[mask], color='gray', lw=0.6, alpha=0.6, label="Noisy")
    ax.plot(f_e[mask]/1000, p_e[mask], '--', color='#1C7293', lw=1.2, label="DCAMF-Net")
    ax.set_xlabel("Frequency (kHz)"); ax.set_ylabel("PSD (dB/Hz)")
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_title("PSD Comparison")
    plt.tight_layout(); plt.show()


---
## Part 4: 基线模型 (CRN → Conv-TasNet → DPRNN)

### 4.1 CRN (时频域)

In [ ]:
adapter_dir = str(PROJECT_ROOT / "baselines" / "CRN-causal" / "adapter")
scripts_dir = str(PROJECT_ROOT / "baselines" / "CRN-causal" / "scripts")

print("Step 1: Run adapter scripts (platform-independent):")
print(f"  cd {adapter_dir}")
print("  python adapter_tr.py")
print("  python adapter_cv.py")
print("  python adapter_tt.py")
print()
print("Step 2: Train CRN:")
print(f"  cd {scripts_dir}")
print("  python train.py --gpu_ids=0 --tr_list=../filelists/tr_list.txt --cv_file=../data/datasets/cv/cv.ex --ckpt_dir=exp --logging_period=5 --lr=0.0002 --batch_size=16 --buffer_size=32 --max_n_epochs=150")
print()
print("Step 3: Inference:")
print(f"  cd {scripts_dir}")
print("  python test.py --gpu_ids=0 --ckpt_dir=exp --model_file=exp/models/best.pt --tt_list=../filelists/tt_list.txt --est_path=../data/estimates")


### 4.2 Conv-TasNet (时域)

In [ ]:
ct_dir = str(PROJECT_ROOT / "baselines" / "conv_tasnet")
print(f"cd {ct_dir} && python train.py")
print("Uses asteroid.models.ConvTasNet - hyperparameters pre-configured.")


### 4.3 DPRNN (时域)

In [ ]:
dp_dir = str(PROJECT_ROOT / "baselines" / "dprnn")
print(f"cd {dp_dir} && python train.py")
print("Uses asteroid.models.DPRNNTasNet - hyperparameters pre-configured.")


### 4.4 四模型汇总表

In [ ]:
# Same table as 3.2 -- already shown above
print("See Part 3.2 for the full comparison table.")
print("Key takeaway:")
print("  DCAMF-Net SDRi=12.84 dB -- BEST among all models")
print("  DCAMF-Net SI-SNRi=12.13 dB -- competitive with Conv-TasNet (13.65)")
print("  Conv-TasNet SDRi=-7.56 dB -- severe nonlinear distortion!")


---
## Part 5: 消融实验

### 5.1 训练三个消融变体

In [ ]:
from pathlib import Path
train_abl_script = str(PROJECT_ROOT / "dcamf_net" / "train_ablation.py")
train_dir = str(PROJECT_ROOT / "data" / "ShipsEar" / "train")

ablations = {
    "No Global Branch": "model_ablation1",
    "No Local Branch": "model_ablation2",
    "No Conv Enhancement": "model_ablation3",
}
for name, model_id in ablations.items():
    save_dir = str(PROJECT_ROOT / "experiments" / "ablation" / model_id / "checkpoints")
    cmd = f"{sys.executable} {train_abl_script} --model {model_id} --train_dir {train_dir} --save_dir {save_dir}"
    print(f"{name}:")
    print(f"  {cmd}")
    print()


### 5.2 消融结果汇总

In [ ]:
abl_results = {
    "Variant": ["DCAMF-Net (Full)", "Without Global Branch", "Without Local Branch", "Without Conv Enhancement"],
    "SI-SNRi (dB)": [12.13, -0.01, 1.40, 10.12],
    "SDRi (dB)": [12.84, -0.02, -18.30, 12.91],
}
df_abl = pd.DataFrame(abl_results)
display(df_abl.style.apply(highlight_best, subset=["SI-SNRi (dB)", "SDRi (dB)"]))

print("\nKey findings:")
print("  Global branch: ESSENTIAL -- removing it kills SI-SNRi (12.13 -> -0.01)")
print("  Local branch: CRITICAL -- removing it destroys SDRi (12.84 -> -18.30)")
print("  Conv enhancement: Smaller impact in 10-block config (12.13 -> 10.12)")


---
## Part 6: 多层掩码融合权重分析

### 6.1 生成高/低 SNR 数据

In [ ]:
print("Generating low SNR data -> data/ShipsEar_low/")
r = subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_data_low.py")],
                   cwd=str(PROJECT_ROOT / "scripts"), capture_output=True, text=True)
print("Done" if r.returncode == 0 else f"Error: {r.stderr}")

print("\nGenerating high SNR data -> data/ShipsEar_high/")
r = subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_data_high.py")],
                   cwd=str(PROJECT_ROOT / "scripts"), capture_output=True, text=True)
print("Done" if r.returncode == 0 else f"Error: {r.stderr}")

for d in ["data/ShipsEar_low", "data/ShipsEar_high", "data/ShipsEar"]:
    p = PROJECT_ROOT / d
    if p.exists() and (p / "train" / "clean").exists():
        n = len(list((p / "train" / "clean").glob("*.wav")))
        print(f"  {d}/ -> {n} training samples")


### 6.2 三组分别训练

In [ ]:
train_script = str(PROJECT_ROOT / "dcamf_net" / "train.py")
low_dir = str(PROJECT_ROOT / "data" / "ShipsEar_low" / "train")
high_dir = str(PROJECT_ROOT / "data" / "ShipsEar_high" / "train")
low_save = str(PROJECT_ROOT / "experiments" / "mask_fusion_weights" / "low")
high_save = str(PROJECT_ROOT / "experiments" / "mask_fusion_weights" / "high")

print("Train on LOW SNR data:")
print(f"  {sys.executable} {train_script} --train_dir {low_dir} --lr 5e-4 --save_dir {low_save} --use_tensorboard")
print()
print("Train on HIGH SNR data:")
print(f"  {sys.executable} {train_script} --train_dir {high_dir} --lr 5e-4 --save_dir {high_save} --use_tensorboard")
print()
print("Average SNR = already trained in Part 2 (standard model)")
print()
print("After training, copy logs:")
print(f"  cp {PROJECT_ROOT}/experiments/dcamf_net/checkpoints/train.log {PROJECT_ROOT}/experiments/mask_fusion_weights/train_avg.log")
print(f"  cp {low_save}/train.log {PROJECT_ROOT}/experiments/mask_fusion_weights/train_low.log")
print(f"  cp {high_save}/train.log {PROJECT_ROOT}/experiments/mask_fusion_weights/train_high.log")


### 6.3 融合权重可视化

In [ ]:
import re

def extract_weights(log_path):
    if not log_path.exists(): return None
    with open(log_path) as f: lines = f.readlines()
    for line in reversed(lines):
        if "MaskFusion Weights (softmax):" in line:
            m = re.search(r'\[(.+?)\]', line)
            if m: return [float(x) for x in m.group(1).split(',')]
    return None

log_dir = PROJECT_ROOT / "experiments" / "mask_fusion_weights"
groups = {"Average SNR": "train_avg.log", "Low SNR": "train_low.log", "High SNR": "train_high.log"}
colors = ['#1C7293', '#999999', '#CCCCCC']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(1, 11)
for i, (name, fname) in enumerate(groups.items()):
    w = extract_weights(log_dir / fname)
    if w and len(w) == 10:
        ax.bar(x + i*0.25, w, 0.25, label=name, color=colors[i], edgecolor='black', linewidth=0.5)
        for j, val in enumerate(w):
            if val > 0.01:
                ax.text(x[j]+i*0.25, val, f'{val:.2f}', ha='center', va='bottom', fontsize=7)
    else:
        print(f"{name}: not found")

ax.set_xlabel("DCAM Block Layer"); ax.set_ylabel("Fusion Weight (softmax)")
ax.set_title("Multi-Layer Mask Fusion Weights under Different SNR Conditions")
ax.set_xticks(x + 0.25); ax.set_xticklabels([str(i) for i in range(1, 11)])
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


---
## Part 7: 结果汇总

### 7.1 全部结果汇总表

In [ ]:
summary_data = {
    "Experiment": [
        "Main - ShipsEar Test-1", "Main - ShipsEar Test-2", "Main - ShipsEar Test-3",
        "Main - DeepShip", "Ablation - No Global", "Ablation - No Local",
        "Ablation - No ConvEnh", "Fusion - Low SNR", "Fusion - High SNR"
    ],
    "SI-SNRi (dB)": [12.13, 9.63, -0.74, 2.76, -0.01, 1.40, 10.12, "<1.0", "N/A"],
    "SDRi (dB)": [12.84, 10.65, 2.62, 5.57, -0.02, -18.30, 12.91, "N/A", "N/A"],
    "Note": [
        "Full DCAMF-Net", "Unseen vessel type", "Unseen noise type",
        "Cross-dataset", "Global branch essential", "Local branch critical",
        "Conv enh smaller impact", "Signal-limited regime", "Single-layer convergence"
    ]
}
df_summary = pd.DataFrame(summary_data)
display(df_summary)
df_summary.to_csv(PROJECT_ROOT / "experiments" / "results_summary.csv", index=False)
print("Saved to experiments/results_summary.csv")


### 7.2 完成清单

In [ ]:
print("\nExperiment Checklist:\n")
for item in [
    "Part 1: Env check, raw data, generate data, preview",
    "Part 2: Model summary, train DCAMF-Net, learning curves",
    "Part 3: Load checkpoint, metrics, audio demo, PSD plot",
    "Part 4: CRN, Conv-TasNet, DPRNN, summary table",
    "Part 5: 3 ablation variants, ablation results table",
    "Part 6: High/Low SNR data, 3-group training, fusion weight plot",
    "Part 7: Full results table, export CSV"
]:
    print(f"  [ ] {item}")
print("\nAll experiments reproducible from this notebook!")
